# Classify Land Use Intensity and Calc Proportions per 1km grid cells


In [1]:
import rasterio
import numpy as np
from rasterio.windows import Window
import gc
from pathlib import Path

# Base directory
base_path = Path("/home/georg/data/LEON_P5_BII")

## Load Data

### multiple in one variable

In [3]:
# Base directory
BASE_DIR = base_path / "EO_data_prep"

# Define all file paths in a structured dictionary
FILES = {
    'dnk': {
        2023: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_dnk_2023.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2023.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2023.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2025_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2023_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif" 
        },
        2022: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_dnk_2022.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2022.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2022.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2022.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2022_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2021: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_dnk_2021.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2021.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2021.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2021_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2020: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_dnk_2020.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2020.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2020.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2020.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2020_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2019: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_dnk_2019.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2019.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2019.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2019_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2018: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_dnk_2018.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2018.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2015_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2018_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        }
    },
    'nld': {
        2023: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_nld_2023.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2023.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2023.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2025_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2023_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2022: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_nld_2022.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2022.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2022.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2022.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2022_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2021: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_nld_2021.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2021.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2021.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2021_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2020: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_nld_2020.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2020.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2020.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2020.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2020_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2019: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_nld_2019.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2019.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2019.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2019_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2018: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban10_pasture_natgrass_nld_2018.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2018.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2015_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2018_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        }
    }
}

# access files like:
# FILES['dnk'][2023]['lc']
# FILES['nld'][2021]['eca']

## Batch process Land Use Intensity

In [4]:
import numpy as np
import rasterio
from rasterio.windows import Window, bounds
import gc
from pathlib import Path

# Assumes base_path and FILES are already defined

##___Configuration___##
version = 'v7'  # Update version as needed
countries = ['dnk', 'nld'] # , 'nld'
years = [2018, 2019, 2020, 2021, 2022, 2023] # 2018, 2019, 2020, 2021, 2022, 

# Paths
OUTPUT_DIR = base_path / "BII_LU_layer" / "Land_use_map" / version
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thresholds
THRESHOLDS = {
    'sv': {'low': 33, 'high': 66},
    'plantation': {'low': 0.2, 'high': 0.4},
    'crop': {'bare_low': 60, 'bare_high': 90, 'swf_high': 1000},
    'pasture': {'low': 0.15, 'high': 0.60},
    'urban': {'nt_thresh1': 2, 'nt_thresh2': 10, 'nt_thresh3': 20, 
              'ghsl_thresh1': 20, 'ghsl_thresh2': 100}
}

NO_DATA_CLASSES = [8, 10, 11, 253, 254, 255]
NO_DATA_VALUE = 255
CHUNK_SIZE = 2000

# ============================================
# Helper Functions
# ============================================

def read_window_from_layer(src, lc_window, lc_transform):
    """
    Read data from auxiliary layer for a given LC window.
    Handles different resolutions by mapping LC bounds to aux layer coordinates.
    Resizes output to match LC window shape.
    """
    from scipy.ndimage import zoom
    
    # Get bounds of LC window in coordinate space
    lc_bounds = bounds(lc_window, lc_transform)
    
    # Map those bounds to auxiliary layer's window
    aux_window = rasterio.windows.from_bounds(*lc_bounds, src.transform)
    
    try:
        data = src.read(1, window=aux_window)
    except Exception as e:
        # If window is out of bounds, return zeros
        data = np.zeros((lc_window.height, lc_window.width))
        return data
    
    # Resize to match LC window shape
    if data.shape != (lc_window.height, lc_window.width):
        scale_y = lc_window.height / data.shape[0]
        scale_x = lc_window.width / data.shape[1]
        data = zoom(data, (scale_y, scale_x), order=0)  # order=0 = nearest neighbor
    
    return data

def process_chunk(lc, aux_data, thresholds):
    """Process a single chunk of land cover data"""
    eca, bare, ghsl, swf, rsd, nt, sv = aux_data
    chunk_intensity = lc.copy().astype(np.uint8)
    
    # Set No Data classes first
    no_data_mask = np.isin(lc, NO_DATA_CLASSES)
    chunk_intensity[no_data_mask] = NO_DATA_VALUE

    # Reclassify class 25. Low RSD = SV indeterminate (24), high RSD = pasture (41)
    mask_25 = (lc == 25)
    if np.any(mask_25):
        lc = lc.copy()
        lc[mask_25 & ((rsd < thresholds['pasture']['low']) | np.isnan(rsd))] = 24
        lc[mask_25 & (rsd >= thresholds['pasture']['low'])] = 41

    # mask_25 = (lc == 25)
    # if np.any(mask_25):
    #     lc = lc.copy()
    #     lc[mask_25 & (rsd < thresholds['pasture']['low'])] = 24
    #     lc[mask_25 & (rsd >= thresholds['pasture']['low'])] = 41
    
    # SV (Semi-Natural Vegetation)
    sv_mask = np.isin(lc, [21, 22, 23, 24])
    if np.any(sv_mask):
        eca_vals = eca[sv_mask]
        t = thresholds['sv']
        intensity = np.where(eca_vals > t['high'], 0, np.where(eca_vals >= t['low'], 1, 2))
        chunk_intensity[sv_mask] = lc[sv_mask] * 10 + intensity
    
    # Plantation
    pl_mask = (lc == 3)
    if np.any(pl_mask):
        sv_vals = sv[pl_mask]
        t = thresholds['plantation']
        intensity = np.where(sv_vals < t['low'], 2, np.where(sv_vals <= t['high'], 1, 0))
        chunk_intensity[pl_mask] = 30 + intensity
    
    # Crop
    crop_mask = np.isin(lc, [6, 7])
    if np.any(crop_mask):
        bare_vals = bare[crop_mask]
        swf_vals = swf[crop_mask]
        t = thresholds['crop']
        minimal = (bare_vals < t['bare_low']) & (swf_vals > t['swf_high'])
        light = ((bare_vals < t['bare_low']) & (swf_vals <= t['swf_high'])) | \
                ((bare_vals < t['bare_high']) & (swf_vals > t['swf_high']))
        intensity = np.where(minimal, 0, np.where(light, 1, 2))
        chunk_intensity[crop_mask] = 70 + intensity
    
    # Pasture
    pasture_mask = (lc == 41)
    if np.any(pasture_mask):
        rsd_vals = rsd[pasture_mask]
        t = thresholds['pasture']
        intensity = np.where(rsd_vals < t['low'], 0, np.where(rsd_vals <= t['high'], 1, 2))
        chunk_intensity[pasture_mask] = 60 + intensity
    
    # Urban
    urban_mask = np.isin(lc, [1, 9, 31])
    if np.any(urban_mask):
        nt_vals = nt[urban_mask]
        ghsl_vals = ghsl[urban_mask]
        t = thresholds['urban']
        
        nt_class = np.where(nt_vals < t['nt_thresh1'], 4,
                           np.where(nt_vals < t['nt_thresh2'], 3,
                                   np.where(nt_vals < t['nt_thresh3'], 2, 1)))
        
        intensity = np.where(
            nt_class == 4, 0,
            np.where(
                (nt_class == 3) & (ghsl_vals < t['ghsl_thresh1']), 0,
                np.where(
                    (nt_class == 3) & (ghsl_vals >= t['ghsl_thresh1']), 1,
                    np.where(
                        (nt_class == 2) & (ghsl_vals < t['ghsl_thresh2']), 1,
                        np.where(
                            (nt_class == 2) & (ghsl_vals >= t['ghsl_thresh2']), 2,
                            np.where(nt_class == 1, 2, 0)
                        )
                    )
                )
            )
        )
        
        chunk_intensity[urban_mask] = 10 + intensity
    
    return chunk_intensity


# ============================================
# Main Processing Loop
# ============================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for country in countries:
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing {country.upper()} {year}")
        print('='*60)
        
        file_paths = FILES[country][year]
        
        print("Starting classification...")
        with rasterio.open(file_paths['lc']) as lc_src:
            profile = lc_src.profile.copy()
            profile.update(compress='lzw', nodata=NO_DATA_VALUE)
            height, width = lc_src.shape
            lc_transform = lc_src.transform
            
            output_file = OUTPUT_DIR / f"lu_intensity_{country}_{year}_{version}.tif"
            
            # Open all auxiliary layers (once)
            aux_sources = {
                'eca': rasterio.open(file_paths['eca']),
                'bare': rasterio.open(file_paths['bare']),
                'ghsl': rasterio.open(file_paths['ghsl']),
                'swf': rasterio.open(file_paths['swf']),
                'rsd': rasterio.open(file_paths['rsd']),
                'nt': rasterio.open(file_paths['nt']),
                'sv': rasterio.open(file_paths['sv']),
            }
            
            try:
                with rasterio.open(output_file, 'w', **profile) as dst:
                    for row_start in range(0, height, CHUNK_SIZE):
                        row_end = min(row_start + CHUNK_SIZE, height)
                        
                        lc_window = Window(0, row_start, width, row_end - row_start)
                        lc = lc_src.read(1, window=lc_window)
                        
                        # Read auxiliary data for this window
                        aux_data = tuple(
                            read_window_from_layer(src, lc_window, lc_transform)
                            for src in [aux_sources['eca'], aux_sources['bare'], 
                                       aux_sources['ghsl'], aux_sources['swf'],
                                       aux_sources['rsd'], aux_sources['nt'], 
                                       aux_sources['sv']]
                        )
                        
                        chunk_intensity = process_chunk(lc, aux_data, THRESHOLDS)
                        dst.write(chunk_intensity, 1, window=lc_window)
                        
                        progress = min((row_end / height) * 100, 100)
                        print(f"  {progress:.1f}%", end='\r')
                        
                        del lc, aux_data, chunk_intensity
                        gc.collect()
                
                print(f"\n✅ Saved: {output_file}")
            
            finally:
                # Close all sources
                for src in aux_sources.values():
                    src.close()
            
            gc.collect()

print("\n🎯 All processing complete!")


Processing DNK 2018
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v7/lu_intensity_dnk_2018_v7.tif

Processing DNK 2019
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v7/lu_intensity_dnk_2019_v7.tif

Processing DNK 2020
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v7/lu_intensity_dnk_2020_v7.tif

Processing DNK 2021
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v7/lu_intensity_dnk_2021_v7.tif

Processing DNK 2022
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v7/lu_intensity_dnk_2022_v7.tif

Processing DNK 2023
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v7/lu_intensity_dnk_2023_v7.tif

Processing NLD 2018
Starting classification...
  100.0%
✅ Saved: /home/georg/data

In [5]:
REMAP = np.zeros(256, dtype=np.uint8)
REMAP[255] = 255
REMAP[10:13] = [1, 2, 3]
REMAP[70:73] = [4, 5, 6]
REMAP[60:63] = [7, 8, 9]
REMAP[30:33] = [10, 11, 12]
REMAP[240:243] = [13, 14, 15]
REMAP[230:233] = [16, 17, 18]
REMAP[220:223] = [19, 20, 21]
REMAP[210:213] = [22, 23, 24]

for country in countries:
    for year in years:
        src_file = OUTPUT_DIR / f"lu_intensity_{country}_{year}_{version}.tif"
        dst_file = OUTPUT_DIR / f"lu_intensity_{country}_{year}_{version}_remap.tif"
        
        with rasterio.open(src_file) as src:
            profile = src.profile.copy()
            profile.update(nodata=0)
            
            with rasterio.open(dst_file, 'w', **profile) as dst:
                for row_start in range(0, src.height, CHUNK_SIZE):
                    window = Window(0, row_start, src.width, min(CHUNK_SIZE, src.height - row_start))
                    data = src.read(1, window=window)
                    dst.write(REMAP[data], 1, window=window)
                    
        print(f"✅ {country} {year}")

print("🎯 Done!")

✅ dnk 2018
✅ dnk 2019
✅ dnk 2020
✅ dnk 2021
✅ dnk 2022
✅ dnk 2023
✅ nld 2018
✅ nld 2019
✅ nld 2020
✅ nld 2021
✅ nld 2022
✅ nld 2023
🎯 Done!


## Create 1km or 100m Grid cell with proportions

### Multiple LU files

In [7]:
##___Configuration___##
version = 'v7'
countries = ['dnk', 'nld']
years = [2018, 2019, 2020, 2021, 2022, 2023]

BLOCK = 10   # 10 pixels = 100m or 100 for 1km grid

# Directories
input_dir = base_path / "BII_LU_layer" / "Land_use_map" / version
output_dir = base_path / "BII_LU_layer" / "Land_use_proportions" / version
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================
# Define intensity classes
# ============================================
# INTENSITY_CLASSES = {
#     'urban_minimal': 10,
#     'urban_light': 11,
#     'urban_intense': 12,
#     'plantation_minimal': 30,
#     'plantation_light': 31,
#     'plantation_intense': 32,
#     'pasture_minimal': 60,
#     'pasture_light': 61,
#     'pasture_intense': 62,
#     'crop_minimal': 70,
#     'crop_light': 71,
#     'crop_intense': 72,
#     'sv_mature_minimal': 210,
#     'sv_mature_light': 211,
#     'sv_mature_intense': 212,
#     'sv_intermediate_minimal': 220,
#     'sv_intermediate_light': 221,
#     'sv_intermediate_intense': 222,
#     'sv_young_minimal': 230,
#     'sv_young_light': 231,
#     'sv_young_intense': 232,
#     'sv_indeterminate_minimal': 240,
#     'sv_indeterminate_light': 241,
#     'sv_indeterminate_intense': 242,
# }

INTENSITY_CLASSES = {
    'urban_minimal': 1,
    'urban_light': 2,
    'urban_intense': 3,
    'crop_minimal': 4,
    'crop_light': 5,
    'crop_intense': 6,
    'pasture_minimal': 7,
    'pasture_light': 8,
    'pasture_intense': 9,
    'plantation_minimal': 10,
    'plantation_light': 11,
    'plantation_intense': 12,
    'sv_indeterminate_minimal': 13,
    'sv_indeterminate_light': 14,
    'sv_indeterminate_intense': 15,
    'sv_young_minimal': 16,
    'sv_young_light': 17,
    'sv_young_intense': 18,
    'sv_intermediate_minimal': 19,
    'sv_intermediate_light': 20,
    'sv_intermediate_intense': 21,
    'sv_mature_minimal': 22,
    'sv_mature_light': 23,
    'sv_mature_intense': 24,
}



# ============================================
# Process all countries and years
# ============================================
for country in countries:
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing {country.upper()} {year}")
        print('='*60)
        
        # Load intensity file
        input_file = input_dir / f"lu_intensity_{country}_{year}_{version}_remap.tif"
        
        if not input_file.exists():
            print(f"⚠️  File not found: {input_file}")
            continue
        
        print(f"Loading: {input_file.name}")
        
        with rasterio.open(input_file) as src:
            intensity = src.read(1)
            height, width = src.shape
            
            # 1km grid dimensions (100 pixels = 1km at 10m resolution)
            grid_height = height // BLOCK
            grid_width = width // BLOCK
            
            print(f"Original: {height} x {width} (10m)")
            print(f"Grid: {grid_height} x {grid_width} (1km)")
            
            # Store profile for output
            transform = src.transform * src.transform.scale(BLOCK, BLOCK)
            output_profile = src.profile.copy()
            output_profile.update({
                'height': grid_height,
                'width': grid_width,
                'transform': transform,
                'dtype': 'float32',
                'count': len(INTENSITY_CLASSES),
                'compress': 'lzw',
                'nodata': -9999
            })
            
            # Calculate proportions for all classes
            print("Calculating proportions...")
            proportion_grids = {}
            
            for class_name, class_value in INTENSITY_CLASSES.items():
                print(f"  {class_name}...", end='')
                
                grid = np.zeros((grid_height, grid_width), dtype=np.float32)
                
                # Aggregate to 1km
                for i in range(grid_height):
                    for j in range(grid_width):
                        row_start = i * BLOCK
                        col_start = j * BLOCK
                        row_end = min(row_start + BLOCK, height)
                        col_end = min(col_start + BLOCK, width)
                        
                        block = intensity[row_start:row_end, col_start:col_end]
                        
                        # Exclude no data (255) from calculations
                        valid_pixels = block[block != 255]
                        
                        if valid_pixels.size > 0:
                            class_pixels = np.sum(valid_pixels == class_value)
                            grid[i, j] = class_pixels / valid_pixels.size
                        else:
                            grid[i, j] = -9999  # No valid data
                
                proportion_grids[class_name] = grid
                
                # Quick stats
                valid_cells = np.sum(grid >= 0)
                non_zero = np.sum(grid > 0)
                print(f" {non_zero}/{valid_cells} cells")
        
        # Save as multi-band GeoTIFF
        output_file = output_dir / f"lu_proportions_{country}_{year}_{version}_{BLOCK}0m.tif"
        
        print(f"\nSaving: {output_file.name}")
        with rasterio.open(output_file, 'w', **output_profile) as dst:
            for i, (class_name, grid) in enumerate(proportion_grids.items(), 1):
                dst.write(grid, i)
                dst.set_band_description(i, class_name)
        
        print(f"✅ Saved with {len(INTENSITY_CLASSES)} bands")

print("\n🎯 All proportions calculated and saved!")


Processing DNK 2018
Loading: lu_intensity_dnk_2018_v7_remap.tif
Original: 35361 x 45093 (10m)
Grid: 3536 x 4509 (1km)
Calculating proportions...
  urban_minimal... 1118355/15908578 cells
  urban_light... 113346/15908578 cells
  urban_intense... 20584/15908578 cells
  crop_minimal... 983178/15908578 cells
  crop_light... 2112751/15908578 cells
  crop_intense... 2454504/15908578 cells
  pasture_minimal... 1447137/15908578 cells
  pasture_light... 304803/15908578 cells
  pasture_intense... 220434/15908578 cells
  plantation_minimal... 1400482/15908578 cells
  plantation_light... 294113/15908578 cells
  plantation_intense... 84877/15908578 cells
  sv_indeterminate_minimal... 8182/15908578 cells
  sv_indeterminate_light... 52274/15908578 cells
  sv_indeterminate_intense... 320418/15908578 cells
  sv_young_minimal... 18211/15908578 cells
  sv_young_light... 119796/15908578 cells
  sv_young_intense... 572121/15908578 cells
  sv_intermediate_minimal... 29270/15908578 cells
  sv_intermediate_l